BALANCE DE MASA  
Para determinar el balance de masa geodesico es necesario tener 2 DEMs, uno inicial y otro final, que abarquen tu glaciar.

PRIMER PASO es abrir tus Rasters en un visor GIS. Si tenes una imagen o puntos de referencia bien georeferenciados te recomiendo que los uses para tu Raster inicial y tu Raster final a esa referencia.  
Para ello, PASO DOS, y en caso de no georeferencies con el software GIS, deberas medir el desplazamiento en X y en Y de tus Rasters respecto a tu referencia.

Como opción b de PASO DOS, si no tienes referencia geográfica precisa o no quieres georreferenciar a referencia ambos Rasters (inicial y final), pero SI notas que un Raster esta desplazado respecto al otro, puedes mover solo un Raster respecto a otro, para ello deberas medir el desplazamiento en X y en Y de un Raster respecto al otro dentro de tu visor GIS.

# CORREGIR HORIZONTALMENTE LOS RASTERS
(entre ellos => modificaras uno solo!!!)  
(ambos a referencia => modificaras ambos!!!)

OJO, buscar las indicaciones ***# MODIFICAR*** porque son las variables y datos que **tiene que cambiar el usuario del código**

### Importamos librerias, definimos funciones necesarias y definimos nuestros directorios

In [ ]:
from osgeo import gdal
import os
import numpy as np
import sys
sys.path.append('../')
import warnings
warnings.filterwarnings('ignore')

In [ ]:
def plote_porcent(banda_del_arreglo, banda = 'banda', p = 0, nodata = None, figsize = (12,6)): 
    maxim = 100 -p
    
    band = banda_del_arreglo
    
    min_b, max_b = np.min(band), np.max(band)
    p_b_min = np.percentile(band, p)
    p_b_max = np.percentile(band, 100-p)
     
    plt.figure(figsize = figsize)
    plt.hist(band.ravel(), bins = 100)
    plt.axvline(p_b_min, color = 'red' ,linestyle = '--', label = f'Percentil {p} %')
    plt.axvline(p_b_max, color = 'black' ,linestyle = '--', label = f'Percenil {maxim}%')
    plt.legend()
    plt.title(f'Histograma {banda}')
    #Como hago para que el titulo me tome 
    plt.show()
    
def porcentajes(banda_del_arreglo, p = 0, nodata = None):    
    banda = banda_del_arreglo
    
    #Mínimo y máximo:
    min_b, max_b = np.min(banda), np.max(banda)

    #Percentiles 2% y 98%
    p_b_min = np.percentile(banda, p)
    p_b_max = np.percentile(banda, 100-p)

    print(f'Mínimo:{min_b}, Máximo:{max_b}')
    print(f'Percentil {p}%: {p_b_min}, Percentil {p}%: {p_b_max}')

In [ ]:
# MODIFICAR direcciorios, segun cual elijas mover (o los 2) (*)

dir_DEM_o = 'C:/' # MODIFICAR

dir_DEM_proc = 'C:/DEM_procesando' # MODIFICAR

DEM_i = 'tu_DEM_fechaInicial' # MODIFICAR
DEM_f = 'tu_DEM_fechaFinal' # MODIFICAR
IMG_i = 'tu_IMG_fechaInicial' # MODIFICAR
IMG_f = 'tu_IMG_fechaFinal' # MODIFICAR
RGBN_f = 'tu_RGBN_fechaInicial' # MODIFICAR
RGBN_i = 'tu_RGBN_fechaFinal' # MODIFICAR

archivo_DEM_i = f'{dir_DEM_o}/{DEM_i}.tif'
archivo_DEM_f = f'{dir_DEM_o}/{DEM_f}.tif'

archivo_DEM_i_2 = f'{dir_DEM_proc}/{DEM_i}_movido.tif'
archivo_DEM_i_2b = f'{dir_DEM_proc}/{DEM_i}_movido_nan.tif'
archivo_DEM_f_2 = f'{dir_DEM_proc}/{DEM_f}_movido.tif'
archivo_DEM_f_2b = f'{dir_DEM_proc}/{DEM_f}_movido_nan.tif'
# uno de los dos (i o f) no se hará
# en este ejemplo vamos a mover f hacia i
archivo_IMG_i = f'{dir_DEM_o}/{IMG_i}.tif'
archivo_IMG_f = f'{dir_DEM_o}/{IMG_f}.tif'
archivo_RGBN_i = f'{dir_DEM_o}/{RGBN_i}.tif'
archivo_RGBN_f = f'{dir_DEM_o}/{RGBN_f}.tif'

archivo_IMG_i_2  = f'{dir_DEM_proc}/{IMG_i}_movida.tif'
archivo_RGBN_i_2 = f'{dir_DEM_proc}/{RGBN_i}_movida.tif'
archivo_IMG_f_2  = f'{dir_DEM_proc}/{IMG_f}_movida.tif'
archivo_RGBN_f_2 = f'{dir_DEM_proc}/{RGBN_f}_movida.tif'

Te recomiendo que vayas indentando con # los archivos que no corregis/moves/usas

# CORREGIR HORIZONTALMENTE
usando el parametro *GeoTransform*

Pero, ¿Qué es cada uno de los **parámetros** del **GeoTransform?**

1) La coordenada superior izquierda en el eje X  
2) Ancho del pixel en X (Este-Oeste)  
3) La rotación de las filas (normalmente esta es 0)  
4) La coordenada superior izquierda en el eje Y  
5) La rotación de las columnas (normalmente esta es 0)  
6) Ancho del pixel en Y (Norte-Sur): En el caso de las imágenes que trabajamos nostros, que apuntan hacia el norte, este siempre toma un valor negativo.  

*Movemos la cantidad de metros que vamos a mover (porque esta en sistema de coord planas).*

### DESPLAZAMIENTO 
(medido en tu visor GIS) => valores a mover

In [ ]:
en_X = 37 # MODIFICAR  según tu medición
en_Y = - 36 # MODIFICAR según tu medición

In [ ]:
lo cambiaste!?

### MOVER IMAGEN DE 1 BANDA

(Si tenes que corregir ambas, solo tiene que cambiar los nombres de los archivos -ver anotaciones: *# MODIFICAR* en el código- y volver a correr el código.)

In [ ]:
IMG = gdal.Open(archivo_IMG_f) # MODIFICAR _f por _i si vas a mover la inicial o para hacer ambas
gt = IMG.GetGeoTransform()
src = IMG.GetProjection()
IMG = IMG.ReadAsArray()
print('Dimensiones de la imagen', IMG.shape)

In [ ]:
print('Parametros de tu GeTransform: ', gt)

**Habiendo determinado de antemano el desplazamiento relativo en X y en Y corregimos el gt:**

In [ ]:
# Desplazamiento en ejes cartesianos
ulx = gt[0] + en_X
uly = gt[3] + en_Y

gt_movida = (ulx, gt[1], gt[2], uly, gt[4], gt[5])

print('Nueva gt: ', gt_movida)

**Exportar IMAGEN movida:**

In [ ]:
# Dimensiones de Filas y Columnas:
filas = IMG.shape[0]
columnas = IMG.shape[1]

# La orthofoto tiene 1 sola banda:
bandas = 1

# Definir extensión de salida:
driver = gdal.GetDriverByName('GTiff')

# Guardar (el float32 lo chequeamos abrindo el DEM original en software GIS)
imagen_salida = driver.Create(f'{archivo_IMG_f_2}', columnas, filas, bandas, gdal.GDT_Float32)  # MODIFICAR _f por _i si vas a mover la inicial o para hacer ambas

# LO MAS IMPORTANTE, Le asignamos el sistema de coordenadas y el nuevo gt a la imagen de salida:
imagen_salida.SetProjection(src)
imagen_salida.SetGeoTransform(gt_movida)
# Podemos corroborar  que se haya asignado correctamente:
imagen_salida.GetGeoTransform()

# Cargarle la banda
imagen_salida.GetRasterBand(1).WriteArray(IMG[:,:])

# Liberamos el archivo con el comando del. y de esa forma se termina de escribir.
del imagen_salida

### MOVER RASTER DE MAS DE 1 BANDA

In [ ]:
RGBN = gdal.Open(archivo_RGBN_f) # MODIFICAR _f por _i si vas a mover la inicial o para hacer ambas
bandas = RGBN.RasterCount
print('Cant. de Bandas: ', bandas)
gt = RGBN.GetGeoTransform()
src = RGBN.GetProjection()
RGBN = RGBN.ReadAsArray()

# Desplazamiento en ejes cartesianos
ulx = gt[0] + en_X
uly = gt[3] + en_Y
gt_movida = (ulx, gt[1], gt[2], uly, gt[4], gt[5])

bandas = RGBN.shape[0]
filas = RGBN.shape[1]
columnas = RGBN.shape[2]

# Definir extensión de salida:
driver = gdal.GetDriverByName('GTiff')
# Guardar (el float32 lo chequeamos abrindo el DEM original en software GIS)
RGBN_salida = driver.Create(archivo_RGBN_f_2, columnas, filas, bandas, gdal.GDT_Float32)  # MODIFICAR _f por _i si vas a mover la inicial o para hacer ambas
RGBN_salida.SetProjection(src)
# Le asignamos el gt modificado y lo corroboramos
RGBN_salida.SetGeoTransform(gt_movida)
print(RGBN_salida.GetGeoTransform())
# Le "cargamos" cada una de las bandas desde el archivo original
RGBN_salida.GetRasterBand(1).WriteArray(RGBN[0,:,:])
RGBN_salida.GetRasterBand(2).WriteArray(RGBN[1,:,:])
RGBN_salida.GetRasterBand(3).WriteArray(RGBN[2,:,:])
#RGBN_salida.GetRasterBand(4).WriteArray(RGBN[3,:,:]) #si tiene 4 bandas
#Liberamos el archivo con el comando del. De esa forma se termina de escribir.
del RGBN_salida

### MOVER DEM

In [ ]:
# Abre el archivo DEM original
DEM = gdal.Open(archivo_DEM_f) # MODIFICAR _f por _i si vas a mover la inicial o para hacer ambas
gt = DEM.GetGeoTransform()
src = DEM.GetProjection()
DEM = DEM.ReadAsArray()

# Desplazamiento en ejes cartesianos
ulx = gt[0] + en_X
uly = gt[3] + en_Y

gt_movida = (ulx, gt[1], gt[2], uly, gt[4], gt[5])

# Dimensiones de Filas y Columnas:
filas = DEM.shape[0]
columnas = DEM.shape[1]
# El DEM tiene 1 sola banda:
bandas = 1
# Definir extensión de salida:
driver = gdal.GetDriverByName('GTiff')
# Guardar (el float32 lo chequeamos abrindo el DEM original en software GIS)
dem_salida = driver.Create(f'{archivo_DEM_f_2}', columnas, filas, bandas, gdal.GDT_Float32)     # MODIFICAR _f por _i si vas a mover la inicial o para hacer ambas
# LO MAS IMPORTANTE, Le asignamos el sistema de coordenadas y el nuevo gt a la imagen de salida:
dem_salida.SetProjection(src)
dem_salida.SetGeoTransform(gt_movida)
# Cargarle la banda
dem_salida.GetRasterBand(1).WriteArray(DEM[:])
# Liberamos el archivo con el comando del. y de esa forma se termina de escribir:
del dem_salida

### MOVER DEM
y que los pixeles 'sin datos' no se transformen en -9999!!!:

In [ ]:
# Abre el archivo DEM original
DEM = gdal.Open(archivo_DEM_f) # MODIFICAR _f por _i si vas a mover la inicial o para hacer ambas
gt = DEM.GetGeoTransform()
src = DEM.GetProjection()
DEM = DEM.ReadAsArray()

# Cambia los valores de -9999 a NaN
DEM = np.where(DEM == -9999, np.nan, DEM)

# Desplazamiento en ejes cartesianos
ulx = gt[0] + en_X
uly = gt[3] + en_Y

gt_movida = (ulx, gt[1], gt[2], uly, gt[4], gt[5])

# Dimensiones de Filas y Columnas:
filas = DEM.shape[0]
columnas = DEM.shape[1]
# El DEM tiene 1 sola banda:
bandas = 1
# Definir extensión de salida:
driver = gdal.GetDriverByName('GTiff')
# Guardar (el float32 lo chequeamos abrindo el DEM original en software GIS)
dem_salida = driver.Create(f'{archivo_DEM_f_2b}', columnas, filas, bandas, gdal.GDT_Float32)    # MODIFICAR _f por _i si vas a mover la inicial o para hacer ambas
# LO MAS IMPORTANTE, Le asignamos el sistema de coordenadas y el nuevo gt a la imagen de salida:
dem_salida.SetProjection(src)
dem_salida.SetGeoTransform(gt_movida)
# Cargarle la banda
dem_salida.GetRasterBand(1).WriteArray(DEM[:])
# Liberamos el archivo con el comando del. y de esa forma se termina de escribir:
del dem_salida

El ejemplo esta hecho con los archivos *final*.

(Si tenes que corregir ambas, solo tiene que cambiar los nombres de los archivos -ver anotaciones: *# MODIFICAR* en el código- y volver a correr el código.)